<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_1_Text_to_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [2]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [3]:
import sqlite3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    age INTEGER,
    marks INTEGER
)
""")

cursor.execute("DELETE FROM students")

students = [
    (1, "Rahul", "CSE", 21, 85),
    (2, "Priya", "ECE", 22, 91),
    (3, "Arjun", "CSE", 20, 78),
    (4, "Sneha", "IT", 21, 88),
    (5, "Kiran", "ECE", 22, 95)
]

cursor.executemany(
    "INSERT INTO students VALUES (?, ?, ?, ?, ?)",
    students
)

conn.commit()

print("Database and students table created successfully!")

Database and students table created successfully!


In [4]:
cursor.execute("SELECT * FROM students")
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'Rahul', 'CSE', 21, 85)
(2, 'Priya', 'ECE', 22, 91)
(3, 'Arjun', 'CSE', 20, 78)
(4, 'Sneha', 'IT', 21, 88)
(5, 'Kiran', 'ECE', 22, 95)


In [5]:
def retrieve_database_schema():

    cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """)

    tables = cursor.fetchall()

    schema = ""

    for table in tables:
        table_name = table[0]

        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()

        schema += f"\nTable: {table_name}\n"

        for column in columns:
            column_name = column[1]
            data_type = column[2]

            schema += f"- {column_name}: {data_type}\n"

    return schema


schema = retrieve_database_schema()

print("Retrieved Database Schema:")
print(schema)

Retrieved Database Schema:

Table: students
- id: INTEGER
- name: TEXT
- department: TEXT
- age: INTEGER
- marks: INTEGER



In [6]:
def generate_sql(question, schema):

    prompt = f"""
You are a SQL expert.

Use the following retrieved database schema:

{schema}

Convert the user's natural-language question into a valid SQLite SQL query.

Return ONLY the SQL query.
Do not provide explanations.
Do not use markdown code blocks.

User Question:
{question}
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    sql_query = interaction.output_text.strip()

    sql_query = sql_query.replace("```sql", "")
    sql_query = sql_query.replace("```", "")
    sql_query = sql_query.strip()

    return sql_query

print("SQL generation function created successfully!")

SQL generation function created successfully!


In [7]:
def execute_sql(sql_query):

    try:
        cursor.execute(sql_query)
        results = cursor.fetchall()
        return results

    except Exception as e:
        return f"SQL Error: {e}"

print("SQL execution function created successfully!")

SQL execution function created successfully!


In [8]:
def text_to_sql_system(question):

    print("========== TEXT-TO-SQL SYSTEM ==========")

    print("\nUser Question:")
    print(question)

    # Step 1: Retrieve database schema
    schema = retrieve_database_schema()

    print("\nRetrieved Schema:")
    print(schema)

    # Step 2: Generate SQL using Gemini
    sql_query = generate_sql(question, schema)

    print("\nGenerated SQL:")
    print(sql_query)

    # Step 3: Execute SQL
    results = execute_sql(sql_query)

    print("\nQuery Result:")

    if isinstance(results, str):
        print(results)
    elif len(results) == 0:
        print("No results found.")
    else:
        for row in results:
            print(row)

In [9]:
text_to_sql_system("Which students have marks greater than 90?")

========== TEXT-TO-SQL SYSTEM ==========

User Question:
Which students have marks greater than 90?

Retrieved Schema:

Table: students
- id: INTEGER
- name: TEXT
- department: TEXT
- age: INTEGER
- marks: INTEGER


Generated SQL:
SELECT name FROM students WHERE marks > 90;

Query Result:
('Priya',)
('Kiran',)


In [10]:
text_to_sql_system("Show the names of students from the CSE department.")

========== TEXT-TO-SQL SYSTEM ==========

User Question:
Show the names of students from the CSE department.

Retrieved Schema:

Table: students
- id: INTEGER
- name: TEXT
- department: TEXT
- age: INTEGER
- marks: INTEGER


Generated SQL:
SELECT name FROM students WHERE department = 'CSE'

Query Result:
('Rahul',)
('Arjun',)
